# Forget-MI LoKU — Machine Unlearning Pipeline

> ✅ **Cell 4 = exp11 FINAL: MULTI-SEED (42,123,7) tại IHL=0.75 — bản HONEST chính thức.**
> Sweep exp11d đã xác nhận IHL=0.75 là sweet-spot; giờ chạy 3 seed → **mean ± std** (con số luận văn).
> Train dừng theo **loss validation** (`early_stop_metric=val`), **KHÔNG đụng `F_re`** (đã gỡ kỹ xảo).
> Colab: **Cell 1** (pull) → *(bỏ qua 2,3 nếu có data)* → **Cell 3.5** (INTEGRITY CHECK) → **Cell 4 → Cell 5**.

> ⚖️ **Hai biến thể (khai báo trong luận văn):**
> - **exp11 (HONEST, chính)**: `distill_teacher=og`, `early_stop_metric=val`, `distill_forget=0` → KHÔNG dùng `F_re` khi train.
> - **exp10c (F_re-allowed, upper-bound để so sánh)**: `distill_teacher=re`, `early_stop_metric=cossim`, `distill_forget>0`.

Notebook thực hiện toàn bộ pipeline:

1. **Cell 1** — Mount Drive + pull code mới (KHÔNG xóa data đã extract)
2. **Cell 2** — Extract data & models (chỉ chạy LẦN ĐẦU)
3. **Cell 3** — Preprocess (chỉ chạy LẦN ĐẦU)
4. **Cell 3.5** — Verify config + **INTEGRITY CHECK** (không dùng F_re khi train)
5. **Cell 4** — Huấn luyện LoKU Unlearning (multi-seed chính thức)
6. **Cell 5** — **Auto-commit & push** kết quả lên GitHub

## Workflow

1. Sửa `config.yaml` ở local → `git push`
2. Colab: **Cell 1** (pull). Lần đầu thêm **Cell 2 → Cell 3** (~3 phút).
3. **Cell 3.5** (verify + integrity) → **Cell 4** → **Cell 5**
4. Local: `git pull` lấy file MD/summary; điền Observations/Conclusion nếu cần; `git push`

> Lần đầu setup Cell 5: tạo `/content/drive/MyDrive/Forget-MI-Project/.git-secrets.json` (xem hướng dẫn trong cell).

In [ ]:
# ====================================
# CELL 1: Kết nối Drive & Pull Code (giữ data, không clone lại)
# ====================================
from google.colab import drive
import os

# 1. Mount Google Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)
else:
    print("✅ Google Drive đã được kết nối!")

# 2. Pull code mới (KHÔNG xóa thư mục → data đã extract được giữ nguyên)
%cd /content
REPO = "Forget-MI-LoKU"
REPO_URL = "https://github.com/nhnhu146/Forget-MI-LoKU.git"

if not os.path.exists(REPO):
    print(f"🔽 Clone lần đầu: {REPO}")
    !git clone {REPO_URL}
else:
    print(f"🔄 Pull code mới (giữ data đã extract)")
    %cd {REPO}
    !git fetch origin
    !git reset --hard origin/master 2>&1 | tail -3
    %cd /content

%cd {REPO}
!git log --oneline -1

# 3. Cài đặt thư viện (chỉ chạy lần đầu hoặc khi cần update)
import importlib.util
need_install = importlib.util.find_spec("peft") is None or importlib.util.find_spec("pydicom") is None
if need_install:
    print("📦 Cài đặt thư viện...")
    !pip install -q pydicom scikit-image wandb pyyaml pandas
    !pip install -q "transformers==4.38.0" "peft==0.10.0" "accelerate==0.27.0"
else:
    print("✅ Thư viện đã cài, bỏ qua.")

# 4. ⚠️ KIỂM TRA GPU — BẮT BUỘC. Không có GPU → mỗi exp chậm ~25-30× (train treo ở Epoch 0).
import torch
if torch.cuda.is_available():
    print(f"\n🟢 GPU OK: {torch.cuda.get_device_name(0)}")
else:
    print("\n" + "!"*60)
    print("🔴 KHÔNG CÓ GPU — Fisher sẽ ~12 phút, train gần như TREO ở Epoch 0!")
    print("   → Colab: Runtime ▸ Change runtime type ▸ Hardware accelerator = GPU (T4) ▸ Reconnect")
    print("   → Free-tier hết hạn mức GPU cũng bị đẩy về CPU (đợi reset / đổi acc / Colab Pro).")
    print("   → ĐỪNG chạy Cell 4 khi còn dòng đỏ này.")
    print("!"*60)

print("\n✅ Môi trường và mã nguồn đã sẵn sàng!")
print("ℹ️  Lần đầu: chạy Cell 2 → Cell 3 (extract data).")
print("ℹ️  Lần sau: bỏ qua Cell 2 + Cell 3, đi thẳng Cell 3.5 → Cell 4 → Cell 5.")

In [14]:
# ====================================
# CELL 2: Giải nén Data & Models
# ====================================
!python setup_data.py

In [15]:
# ====================================
# CELL 3: Tiền xử lý & Thiết lập Output
# ====================================
# CHỈ CHẠY LẦN ĐẦU (sau đó cache features được giữ trong /content/.../data/metadata/)
import os
import shutil

# 1. Tạo all_data.tsv từ các file báo cáo (idempotent)
!python make_tsv.py

# 2. Cache features — KHÔNG xóa nữa (xóa = phải regenerate 5+ phút mỗi lần)
#    Uncomment 2 dòng dưới nếu bạn THỰC SỰ muốn force regenerate cache
# !rm -f ./data/metadata/cachedfeatures_train_seqlen-*
# !rm -f ./data/metadata/cachednoisyfeatures_train_seqlen-*

# 3. Kết nối thư mục Output với Drive để lưu bền vững
DRIVE_RESULTS = "/content/drive/MyDrive/Forget-MI-Project/unlearning_output"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

if os.path.exists("unlearning_output"):
    if os.path.islink("unlearning_output"):
        os.unlink("unlearning_output")
    else:
        shutil.rmtree("unlearning_output")

!ln -s "{DRIVE_RESULTS}" ./unlearning_output

# 4. Verify cache files có sẵn không
cache_dir = "./data/metadata"
has_cache = False
if os.path.exists(cache_dir):
    files = os.listdir(cache_dir)
    has_cache = any(f.startswith(("cachedfeatures_train_seqlen", "cachednoisyfeatures_train_seqlen"))
                    for f in files)

print(f"\n✅ Output sẽ được lưu tại: {DRIVE_RESULTS}")
print(f"{'✅' if has_cache else '⚠️ '} Cache features {'đã có' if has_cache else 'CHƯA có'} trong {cache_dir}")
if not has_cache:
    print("   → Cell 4 lần đầu sẽ chậm (~5 phút regenerate features). Lần sau sẽ load cache nhanh.")

In [ ]:
# ====================================
# CELL 3.5: Verify config — confirm code mới nhất từ GitHub
# ====================================
# Chạy cell này TRƯỚC Cell 4 để chắc chắn config đúng với exp đang định chạy.

print("📋 Config hiện tại (các tham số hay đổi giữa các exp):\n")
!grep -E "^\s*(forget_margin|alpha|beta|theta|gamma|lora_r|unlearn_epochs|learning_rate|kappa_cls_retain|kappa_cls_forget|distill_teacher|distill_retain_weight|distill_forget_weight|loku_subtract_scale|ihl_forget_weight|lora_image_last_k_blocks|loku_image_subtract_scale|early_stop_metric|eta_re_anchor):" -A 1 config.yaml | grep -v "^--"

print("\n🧼 INTEGRITY CHECK — KHÔNG được dùng F_re khi train (bản honest exp11):")
!grep -E "^\s*(distill_teacher|early_stop_metric|distill_forget_weight|eta_re_anchor):" -A 1 config.yaml | grep "value:" | xargs -I{} echo "   → {}"
print("   ✅ ĐÚNG khi: distill_teacher=og, early_stop_metric=val, distill_forget_weight=0, eta_re_anchor=0")
print("   ⚠️  Nếu distill_teacher=re HOẶC early_stop_metric=cossim → đang DÙNG F_re (chỉ hợp lệ cho exp10c).")

print("\n🧪 Code có cơ chế honest (teacher=og + early-stop val):")
!grep -c "use_og_teacher\|early_stop_metric\|val_subset\|resolve_image_targets\|_fila_decompose\|TRUE-SUBTRACTION" training/forgetmi_loku.py | xargs -I{} echo "   → khớp pattern: {} (nên >= 6)"

print("\n🔍 Git commit đang chạy:")
!git log --oneline -1

print("\n👉 Nếu config CŨ → local `git push` rồi rerun Cell 1.")

In [ ]:
# ====================================
# CELL 4: MULTI-SEED CHÍNH THỨC — exp11 honest, IHL=0.75 (no F_re)
# ====================================
# ✅ Chốt: chạy 3 seed tại config sweet-spot (IHL=0.75, early-stop=val, teacher=og) → mean±std.
#    Đây là con số chính thức cho luận văn. Chỉ bấm chạy (~0.6h).
import os, numpy as np, pandas as pd

EXP_NAME = "exp11_final_ihl075"
SEEDS    = [42, 123, 7]
HYPOTHESIS = ("exp11 FINAL (honest, no F_re): distill_teacher=og, early_stop=val, distill_forget=0, "
              "IHL=0.75 (sweet-spot tu sweep exp11d), image-FILA scale0.3. Multi-seed de bao cao mean+-std. "
              "Da kiem chung ket qua bat bien voi tieu chi early-stop -> khong ky xao.")

CSV = "unlearning_output/results_summary.csv"
if os.path.exists(CSV):
    os.remove(CSV)

for i, s in enumerate(SEEDS):
    print(f"\n{'='*60}\n🎲 SEED {s}  ({i+1}/{len(SEEDS)})\n{'='*60}")
    cmd = (f'PYTHONPATH=. WANDB_MODE=disabled python training/forgetmi_loku.py '
           f'--config config.yaml --fresh --seed {s} '
           f'--exp {EXP_NAME}_seed{s} --hypothesis "{HYPOTHESIS}"')
    get_ipython().system(cmd)

# ----- Tổng hợp mean ± std -----
print(f"\n{'='*64}\n📊 MULTI-SEED FINAL (IHL=0.75, honest)  seeds={SEEDS}\n{'='*64}")
df = pd.read_csv(CSV); df = df[df['seed'].isin(SEEDS)]
metrics = [('MIA','MIA_persample ↓'), ('MIA_paper','MIA_paper ↓'),
           ('forget_ce','forget_ce'), ('test_ce','test_ce'),
           ('Df_AUC','Forget AUC ↓'), ('Df_F1','Forget F1 ↓'),
           ('Dt_AUC','Test AUC ↑'), ('Dt_F1','Test F1 ↑')]
hdr = f"{'Metric':<16}" + "".join(f"{f'seed{s}':>9}" for s in SEEDS) + f"{'mean ± std':>16}"
print(hdr); print("-" * len(hdr))
md_rows = ["| Metric | " + " | ".join(f"seed {s}" for s in SEEDS) + " | **mean ± std** |",
           "|" + "---|" * (len(SEEDS) + 2)]
for key, label in metrics:
    if key not in df.columns:
        continue
    vals = [float(df[df['seed'] == s][key].iloc[-1]) for s in SEEDS if (df['seed'] == s).any()]
    if not vals:
        continue
    m, sd = float(np.mean(vals)), float(np.std(vals))
    print(f"{label:<16}" + "".join(f"{v:>9.3f}" for v in vals) + f"{m:>10.3f}±{sd:.3f}")
    md_rows.append(f"| {label} | " + " | ".join(f"{v:.3f}" for v in vals) + f" | **{m:.3f} ± {sd:.3f}** |")

out_md = f"experiments/{EXP_NAME}_multiseed_summary.md"
with open(out_md, "w", encoding="utf-8") as f:
    f.write(f"# FINAL multi-seed (honest, no F_re, IHL=0.75) — {EXP_NAME}\n\nSeeds: {SEEDS}\n\n"
            + "\n".join(md_rows)
            + "\n\n**Paper (3%)**: MIA=0.571 | Df_AUC=0.735 | Df_F1=0.393 | Dt_AUC=0.625 | Dt_F1=0.250  \n"
            + "**Gold retrained**: MIA=0.000 | Df_AUC=0.566 | Df_F1=0.310 | Dt_AUC=0.626 | Dt_F1=0.362\n")
print(f"\n💾 Summary: {out_md}\n👉 Đây là CON SỐ CHÍNH THỨC. Chạy CELL 5 để push.")

In [18]:
# ====================================
# CELL 5: Auto-commit & push experiment results lên GitHub
# ====================================
# 3 cách setup credentials (chọn 1, theo độ tiện):
#
# CÁCH A — Lưu vào Drive (KHUYẾN NGHỊ, setup 1 lần dùng mãi):
#   Tạo file /content/drive/MyDrive/Forget-MI-Project/.git-secrets.json
#   với nội dung:
#   {
#     "GITHUB_TOKEN": "ghp_xxxxxxxxxxxx",
#     "GIT_EMAIL": "ban@gmail.com",
#     "GIT_NAME": "Nguyen Hoang Nhu"
#   }
#   Tạo token tại: https://github.com/settings/tokens (scope: repo)
#
# CÁCH B — Colab Secrets (CHỈ web colab.research.google.com):
#   Click 🔑 ở sidebar → Add secret: GITHUB_TOKEN, GIT_EMAIL, GIT_NAME
#
# CÁCH C — Nhập tay mỗi session (lazy, không cần setup):
#   Bỏ qua A và B → cell sẽ tự hỏi token mỗi lần chạy
# ===========================================================
import os, json, getpass
from pathlib import Path

GITHUB_REPO = "nhnhu146/Forget-MI-LoKU"
BRANCH = "master"

def load_secrets():
    # CÁCH A — file trên Drive
    drive_path = Path("/content/drive/MyDrive/Forget-MI-Project/.git-secrets.json")
    if drive_path.exists():
        s = json.loads(drive_path.read_text())
        print(f"🔑 Đã load credentials từ {drive_path}")
        return s.get('GITHUB_TOKEN'), s.get('GIT_EMAIL'), s.get('GIT_NAME')

    # CÁCH B — Colab Secrets (web Colab)
    try:
        from google.colab import userdata
        t = userdata.get('GITHUB_TOKEN')
        if t:
            print("🔑 Đã load credentials từ Colab Secrets")
            return t, userdata.get('GIT_EMAIL'), userdata.get('GIT_NAME')
    except Exception:
        pass

    # CÁCH B2 — environment variables
    if os.environ.get('GITHUB_TOKEN'):
        print("🔑 Đã load credentials từ env vars")
        return (os.environ['GITHUB_TOKEN'],
                os.environ.get('GIT_EMAIL', ''),
                os.environ.get('GIT_NAME', ''))

    # CÁCH C — nhập tay (fallback)
    print("🔑 Nhập credentials thủ công (sẽ ẩn khi gõ token):")
    print("   (lần sau muốn auto, tạo file Drive theo CÁCH A ở comment trên)")
    t = getpass.getpass("  GitHub token (ghp_...): ").strip()
    e = input("  Git email: ").strip()
    n = input("  Git name:  ").strip()
    return t, e, n


TOKEN, EMAIL, NAME = load_secrets()

if TOKEN and EMAIL and NAME:
    # 1. Configure git identity (chỉ trong repo này, không ảnh hưởng global)
    !git config user.email "{EMAIL}"
    !git config user.name "{NAME}"

    # 2. Inject token vào remote URL (chỉ trong session này)
    !git remote set-url origin https://{TOKEN}@github.com/{GITHUB_REPO}.git

    # 3. Pull trước để tránh conflict
    !git pull --rebase origin {BRANCH} 2>&1 | tail -5

    # 4. Stage CHỈ file experiment
    !git add experiments/ 2>/dev/null

    # 5. Hiển thị thay đổi
    changes = !git diff --cached --name-only
    if changes and any(c.strip() for c in changes):
        print("\n📦 Files sẽ commit:")
        for f in changes:
            if f.strip():
                print(f"   - {f}")

        commit_msg = f"exp {EXP_NAME}: auto-tracked results"
        !git commit -m "{commit_msg}"
        !git push origin {BRANCH}

        print(f"\n✅ Đã push lên GitHub")
        print(f"🔗 Xem online: https://github.com/{GITHUB_REPO}/tree/{BRANCH}/experiments")
    else:
        print("ℹ️  Không có file experiment mới để commit.")
else:
    print("⚠️  Thiếu credentials — bỏ qua push. Setup theo CÁCH A/B/C ở comment trên.")